# FA-SwinLite: Frequency-Aware Lightweight Transformer for Image Super-Resolution
## ECE 531 Computer Vision — Term Project
**Author:** Melisa ÇOLAK

This notebook trains and evaluates FA-SwinLite along with ablation studies addressing the instructor's feedback:
1. Frequency-band error analysis (gap analysis)
2. Balanced loss formulation (λ_pix, λ_perc, λ_freq)
3. Ablation separating lightweight arch vs frequency loss
4. Identical parameter-budget comparison

---
> **AI Acknowledgment:** Claude (Anthropic) was used to assist with code structure and writing. All content has been reviewed and verified by the author.

---

## 0. Setup & Install

In [ ]:
# Install dependencies
!pip install lpips --quiet
!pip install torchmetrics --quiet

In [ ]:
# Clone or upload project files
# Option A: If using GitHub (recommended)
# !git clone https://github.com/YOUR_USERNAME/fa-swinlite.git
# %cd fa-swinlite/src

# Option B: Upload the src/ folder to Colab manually, then:
import os
# If files are in /content/src:
# os.chdir('/content/src')

# For now, assume files are in current directory or src/
import sys
if os.path.exists('src'):
    sys.path.insert(0, 'src')
print('Path set up.')

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Download DIV2K Dataset

In [ ]:
from dataset import download_div2k

DATA_ROOT = './data/DIV2K'
# Downloads HR + LR (bicubic x2 and x4) — ~7 GB total
# This will take ~10-20 min on Colab
download_div2k(root=DATA_ROOT, scales=[2, 4])
print('Dataset ready!')

## 2. Model Sanity Check

In [ ]:
from model import FASwinLite

print('Model parameter counts:\n')
for scale in [2, 4]:
    for lw in [True, False]:
        m = FASwinLite(scale=scale, lightweight=lw)
        x = torch.randn(1, 3, 64, 64)
        y = m(x)
        label = 'Lightweight (proposed)' if lw else 'Full-size (baseline)'
        print(f'  Scale x{scale} | {label:28s} | '
              f'Params: {m.count_parameters()/1e6:.2f}M | '
              f'Output: {tuple(y.shape)}')

## 3. Loss Function Demo

In [ ]:
from losses import CombinedLoss

criterion = CombinedLoss(lambda_pix=1.0, lambda_perc=0.1, lambda_freq=0.1,
                         use_freq=True, device=DEVICE)

pred   = torch.rand(2, 3, 128, 128).to(DEVICE)
target = torch.rand(2, 3, 128, 128).to(DEVICE)
losses = criterion(pred, target)

print('Loss breakdown:')
for k, v in losses.items():
    val = v.item() if hasattr(v, 'item') else v
    print(f'  {k:8s}: {val:.6f}')
print(f'\nFormula: L_total = 1.0 * L1 + 0.1 * L_perc + 0.1 * L_freq')

## 4. Ablation Study (4 configurations)

| Config | Lightweight Arch | Freq Loss |
|--------|:---:|:---:|
| Baseline_Full_NoFreq | ❌ | ❌ |
| Baseline_Full_Freq   | ❌ | ✅ |
| FA_SwinLite_NoFreq   | ✅ | ❌ |
| **FA_SwinLite_Full** (proposed) | ✅ | ✅ |

Separates the contribution of: (a) lightweight architecture, (b) frequency loss.

In [ ]:
from config import Config

# ── Colab-friendly settings ──────────────────────────────────────────────────
# Change to Config.NUM_EPOCHS=100 for full training
NUM_EPOCHS_QUICK = 30   # quick test
SCALE = 2               # change to 4 for ×4 experiments

print(f'Ablation configs: {len(Config.ABLATION_CONFIGS)}')
for name, lw, freq in Config.ABLATION_CONFIGS:
    print(f'  {name:35s} | lightweight={lw} | freq_loss={freq}')

In [ ]:
from dataset import get_loaders
from train import run_ablation

train_loader, valid_loader = get_loaders(scale=SCALE, root=DATA_ROOT)

ablation_results = run_ablation(
    scale        = SCALE,
    train_loader = train_loader,
    valid_loader = valid_loader,
    device       = DEVICE,
    num_epochs   = NUM_EPOCHS_QUICK,
    results_dir  = './results',
)

## 5. λ_freq Sweep (Frequency Loss Weight Analysis)

In [ ]:
from train import run_lambda_freq_sweep

sweep_results = run_lambda_freq_sweep(
    scale          = SCALE,
    train_loader   = train_loader,
    valid_loader   = valid_loader,
    device         = DEVICE,
    lambda_values  = [0.0, 0.05, 0.1, 0.5, 1.0],
    num_epochs     = 20,   # shorter for sweep
    results_dir    = './results',
)

## 6. Frequency-Band Error Analysis (Gap Analysis for Paper)

In [ ]:
from config import Config
from model import FASwinLite
from evaluate import Evaluator
import torch

# Load all trained models
trained_models = {}
for name, use_lw, use_freq in Config.ABLATION_CONFIGS:
    ckpt_path = f'./checkpoints/{name}_x{SCALE}_best.pth'
    model = FASwinLite(scale=SCALE, lightweight=use_lw).to(DEVICE)
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['state_dict'])
        print(f'  Loaded: {name}')
    trained_models[name] = model

# Run band analysis
evaluator = Evaluator(device=DEVICE)
band_results = evaluator.frequency_band_analysis(trained_models, valid_loader, max_samples=30)

print('\nFrequency-Band Error Analysis:')
print(f'{"Model":35s} {"Low":>10} {"Mid":>10} {"High":>10}')
print('-' * 68)
for name, bands in band_results.items():
    print(f'{name:35s} {bands["low"]:10.4f} {bands["mid"]:10.4f} {bands["high"]:10.4f}')

## 7. Generate All Paper Figures

In [ ]:
import os, json
from visualize import (
    plot_freq_band_analysis,
    plot_ablation_bars,
    plot_psnr_vs_params,
    plot_lambda_sweep,
    plot_training_curves,
    plot_visual_comparison,
)

FIGURES_DIR = './figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Fig 1: Frequency-band gap analysis
plot_freq_band_analysis(
    band_results,
    f'{FIGURES_DIR}/fig1_freq_band_analysis_x{SCALE}.png'
)

# Fig 2: Ablation PSNR bars
plot_ablation_bars(
    ablation_results,
    f'{FIGURES_DIR}/fig2_ablation_bars_x{SCALE}.png',
    scale=SCALE
)

# Fig 3: PSNR vs Params
plot_psnr_vs_params(
    ablation_results,
    f'{FIGURES_DIR}/fig3_psnr_vs_params_x{SCALE}.png',
    scale=SCALE
)

# Fig 4: Lambda sweep
plot_lambda_sweep(
    sweep_results,
    f'{FIGURES_DIR}/fig4_lambda_sweep_x{SCALE}.png',
    scale=SCALE
)

# Fig 5: Training curves
log_paths = {
    name: f'./checkpoints/{name}_x{SCALE}_log.json'
    for name, _, _ in Config.ABLATION_CONFIGS
}
plot_training_curves(
    log_paths,
    f'{FIGURES_DIR}/fig5_training_curves_x{SCALE}.png',
    scale=SCALE
)

# Fig 6: Visual comparison
result_dirs = {
    name: f'./results/{name}'
    for name, _, _ in Config.ABLATION_CONFIGS
    if os.path.exists(f'./results/{name}')
}
if result_dirs:
    plot_visual_comparison(
        result_dirs,
        f'{FIGURES_DIR}/fig6_visual_comparison_x{SCALE}.png',
    )

print('\nAll figures saved to', FIGURES_DIR)

## 8. Display Figures Inline

In [ ]:
from IPython.display import Image as IPImage, display
import glob

for fig_path in sorted(glob.glob(f'{FIGURES_DIR}/*.png')):
    print(f'\n--- {os.path.basename(fig_path)} ---')
    display(IPImage(fig_path, width=800))

## 9. Final Results Summary Table

In [ ]:
from evaluate import print_results_table
print(f'\n=== ABLATION STUDY RESULTS (×{SCALE}) ===')
print_results_table(ablation_results)

# Also save a combined JSON for the paper
import json
with open(f'./results/all_results_x{SCALE}.json', 'w') as f:
    json.dump({
        'ablation':     ablation_results,
        'lambda_sweep': sweep_results,
        'freq_bands':   band_results,
    }, f, indent=2)
print('Results saved.')

## 10. Repeat for ×4 Scale
Run cells 4–9 with `SCALE = 4`.

In [ ]:
# Uncomment to run ×4 experiments
# SCALE = 4
# train_loader_4, valid_loader_4 = get_loaders(scale=4, root=DATA_ROOT)
# ablation_results_4 = run_ablation(4, train_loader_4, valid_loader_4, DEVICE, NUM_EPOCHS_QUICK)
# sweep_results_4    = run_lambda_freq_sweep(4, train_loader_4, valid_loader_4, DEVICE, num_epochs=20)
# band_results_4     = evaluator.frequency_band_analysis(trained_models, valid_loader_4, max_samples=30)